In [1]:
from IPython.display import Image

- SWE-Vision: A Minimal Agent for Advancing Visual Intelligence (Agentic Vision)
    - https://unipat.ai/blog/SWE-Vision 
    - https://github.com/UniPat-AI/SWE-Vision
    - kimi k2.5: zero vision sft, gemini 3 flash agentic vision
- ci (code interpreter) sandbox
    - docker 容器 + jupyter kernel: 有状态的 ci
        - ipython repl
        - 熟悉 docker（镜像/容器），jupyter kernel 实现一个有状态的 ci（进行通信）
    - 有状态的 ci 环境
        - codex: `write_stdin`
        - coding agent（cursor）：持续地跟一个环境交互，send & recieve
- `code_execution` 对比
    - openai api
        - https://developers.openai.com/api/docs/guides/tools-code-interpreter
        - run Python code in a sandboxed environment
        - A container is a fully sandboxed virtual machine that the model can run Python code in. 
    - gemini api
        - https://ai.google.dev/gemini-api/docs/code-execution?hl=zh-cn
        - https://aistudio.google.com/apps/bundled/gemini_visual_thinking?e=0&showAssistant=true&showCode=true

### architecture

In [3]:
Image(url='./figs/swe-vision.png', width=400)

> swe_vision/kernel.py

- swe-vision
    - Docker container（Docker 容器）
    - 轻量级的“虚拟机”或“安全沙箱”。
- Jupyter kernel / IPython kernel（Jupyter/IPython 内核）
- Persistent（持久化的 / 有状态的）
    - 如果大模型每次写代码，系统都新开一个进程跑，那上一步加载的图片、算好的数据就全丢了。 因为是 Persistent（有状态的），大模型可以像人类一样分步工作：
        - 第一步：写代码 img = Image.open('test.png') （加载图片到内存）
        - 第二步：写代码 img.crop(...) （直接使用上一步的 img 变量，因为它还在内存里）
    - 这也是它和普通 `subprocess.run("python xxx.py")` 的本质区别。
- Host connects via jupyter_client（宿主机通过 jupyter_client 连接）
    - 它是什么：Host（宿主机）就是当前运行这个项目的电脑环境。jupyter_client 是一个官方的 Python 库，用来和 Jupyter 内核对话。
    - 在项目中的作用：大模型生成了一段代码后，电脑上的主程序（Host）就会使用 jupyter_client 这个工具，通过 ZMQ 端口（对讲机），把代码发送给 Docker 里面的内核。等内核执行完了，jupyter_client 再把结果接收回来，喂给大模型看。

- swe_vision/cli.py: main entry
    - `python -m swe_vision.cli --image <图片> "<问题>"`
- swe_vision/agent.py：agent loop，负责发给模型、接收 tool call、执行代码、回传结果。
    - swe_vision/config.py:
        - system prompt
        - TOOLS schema
            - execute_code
            - finish
    - run <- run_loop <-
        - `result = await self.kernel.execute(code)`
- **swe_vision/kernel.py**：管理 Docker 容器里的 Jupyter kernel (通信的建立)。
    - `self.kernel = JupyterNotebookKernel()`
- misc
    - swe_vision/file_manager.py：把用户上传的图片复制到宿主机挂载目录，再映射到容器里的 /mnt/data。
    - swe_vision/trajectory.py：把每次运行的对话、代码、输出图片保存成 trajectory。
    - apps/web_app.py：Flask Web UI，支持流式展示推理、代码执行和结果。
    - apps/trajectory_viewer.py：查看历史运行轨迹的可视化页面。

### 启动

#### docker

- `docker build -t swe-vision:latest -f env/Dockerfile env`
    - 构建镜像
- `docker images | grep swe-vision`
    - 镜像是否存在
- `docker ps -a --filter "ancestor=swe-vision:latest"`

#### docker、kernel、jupyter client

- docker container 的 smoke test （health check）
    - 大语言模型（LLM）的智能体（Agent）框架（如 SWE-agent）中，用于在将代码执行权限交给 AI 之前，确保底层的“代码执行沙箱”处于可用状态。
    - --rm：容器退出后自动删除。
        - 用一次性实验台做测试
    - -i (Interactive) 保持标准输入（STDIN）开启，这是为了接收后续的代码流。
    - `<<'PY' ... PY`：shell 的 here-document（heredoc）
    - 在 一个临时 Docker 容器里启动一个 Jupyter 内核（kernel），再用客户端去连接它、等待它就绪，若成功就打印 KERNEL_MANAGER_OK，最后无论成功失败都把资源清理干净。

```shell
docker run --rm -i swe-vision:latest python - <<'PY'
from jupyter_client import KernelManager
km = KernelManager()
km.start_kernel()
kc = km.blocking_client()
kc.start_channels()
try:
    kc.wait_for_ready(timeout=10)
    print('KERNEL_MANAGER_OK')
finally:
    try:
        kc.stop_channels()
    finally:
        km.shutdown_kernel(now=True)
PY
```

```shell
".venv/bin/python" - <<'PY'
import asyncio
from swe_vision.kernel import JupyterNotebookKernel

async def main():
    kernel = JupyterNotebookKernel()
    try:
        await kernel.start()
        result = await kernel.execute("print('kernel_ok')\n1 + 1")
        print(result)
    finally:
        await kernel.shutdown()

asyncio.run(main())
PY
```

#### 运行

```shell
export OPENAI_API_KEY="sk-or-v1-xx" # set your openai api key here
export OPENAI_BASE_URL="https://openrouter.ai/api/v1" # set your openai/openrouter/... base url here
export MODEL_NAME="${MODEL_NAME:-openai/gpt-5.2}" # set your model name here
```
- python apps/web_app.py --port 8080
- testcase
    - `Please help the ant walk through the maze in the upload image. Do not cross walls. draw path in the image`

### 状态管理与 agent loop

- agent.py 决定要执行一段 Python
- 调 `kernel.execute(code)` (JupyterNotebookKernel)
- 如果 kernel 还没启动，就先：
    - 构建/使用 Docker 镜像 (image)
    - 启动容器 (docker)
    - 在容器里启动 ipykernel_launcher
    - 在宿主机上用 jupyter_client 连上它
- 然后发送代码执行请求
- 从 iopub / shell 等通道收回输出
    - zmq ports
- 把文本和图像整理后交回 agent

```
docker build -t swe-vision:latest -f ./env/Dockerfile ./env

docker run -d \
  --name vlm-jupyter-xxxx \
  -p 65500:65500 \
  -p 65501:65501 \
  -p 65502:65502 \
  -p 65503:65503 \
  -p 65504:65504 \
  -v "/Users/you/tmp/vlm_docker_workdir/20260403_135657:/mnt/data" \
  -w /mnt/data \
  swe-vision:latest \
  sleep infinity

docker exec -d vlm-jupyter-xxxx \
  bash -c "python -m ipykernel_launcher -f /mnt/data/.kernel_connection.json --IPKernelApp.matplotlib='inline'"

# cleanup
docker stop -t 5 vlm-jupyter-xxxx      # Stopping Docker container 
docker rm -f vlm-jupyter-xxxx          # Container removed.
```

- llm 如何感知是一个 repl 的交互式 stateful 环境
    - system prompt
    - execute_code 的 description 中

### 交互式测试

```
python - <<'PY'                                   
import asyncio
from swe_vision.kernel import JupyterNotebookKernel
async def main():
    kernel = JupyterNotebookKernel()
    try:
        await kernel.start()
        print(await kernel.execute("x = 10\nprint('x=', x)"))
        print(await kernel.execute("x += 5\nprint('x now=', x)"))
        print(await kernel.execute("import math\nprint(math.sqrt(x))"))
    finally:
        await kernel.shutdown()
asyncio.run(main())
PY
```

- jupyter 就是两部分：
    - 一个长期存活的 kernel 进程
    - 一个不断往它发执行请求的 client

```
# uv init 会自动生成 pyproject.toml 文件。
uv init kernel-demo

cd kernel-demo

uv add jupyter_console ipykernel

# uv run python -m jupyter_console
uv run jupyter console


# jupyter console 是“前端”
# ipykernel_launcher 是“后端”
uv run python -m ipykernel_launcher -f /tmp/kernel_test.json
uv run jupyter console --existing /tmp/kernel_test.json
```

#### jupyter console vs. ipython

> 单体应用 vs 分布式前端
- IPython 一体机
    - 原生的、高度强化的 Python 交互式命令行环境（REPL，Read-Eval-Print Loop）。
    - 当在终端直接运行 ipython 时，界面进程与计算进程是同一个物理进程。它直接在当前进程的内存空间中接管代码解析、执行逻辑并渲染高亮输出。
    - $\text{Input}_{text} \xrightarrow{\text{ast.parse}} \text{Eval}_{memory} \xrightarrow{\text{sys.stdout}} \text{Output}_{render}$
- Jupyter Console “瘦客户端（Thin Client）”
    - 它本身只有显示和输入功能，必须通过网线（ZeroMQ 网络协议）连接到远端的计算主机（Kernel）才能工作。
    - jupyter console 没有任何代码执行能力，纯粹是一个基于终端的前端展示层（Frontend）。它强制遵循 Jupyter 的 C/S（客户端/服务端）架构。
    - 当开发者执行 jupyter console 时，实际上在底层发生了两件事：
        - 它隐式地在后台拉起了一个 ipykernel 进程（即服务端）。
        - 它通过网络端口连接到这个 Kernel，将终端用户的输入序列化打包，通过网络发送给 Kernel 执行，再接收结果进行渲染。
    - $\text{Client} \xleftrightarrow[\text{HMAC Auth}]{\text{ZeroMQ (JSON / TCP Ports)}} \text{Kernel} ( \mathbb{K} )$
- jupyter notebook / jupyter lab
    - `jupyter lab --port xxxx --no-browser --ContentsManager.allow_hidden=True`

### questions

- 并发，多用户请求：资源管理
- 更匹配的 system prompt